# SNA Analysis: Instagram Social Network

**Input:** Output dari noebook `01_EDA.ipynb`
- `post_comment_rank.csv` → edge table (commenter → post_owner)
- `user_comments_received.csv`    → in-degree proxy per post owner
- `post_comment_rank.csv`         → post-level comment rank
- `dataset_final_clean_revised.csv`             → raw dataset untuk node attributes

---

## Edge Architecture

```
PRIMARY  →  comment-on-post
            commenter_username → post_owner_username
            weight = comment_count

LAYER 2  →  mention (opsional, boost weight jika ada)

SKIP     →  co-keyword sebagai edge (terlalu noisy)
            → disimpan sebagai node attribute active_keywords
```

## Centrality yang dihitung

| Metric | Interpretasi Bisnis |
|---|---|
| **In-degree** | Berapa unique commenter datang ke akun ini → popularitas mentah |
| **PageRank** | Komentar dari akun besar bobotnya lebih tinggi → kualitas engagement |
| **Betweenness** | Jembatan antar komunitas → paling valuable untuk brand outreach |
| **Out-degree** | Seberapa aktif akun ini komentar ke akun lain |
| **Authority (HITS)** | Dipilih oleh hub-hub besar → legitimasi |
| **Eigenvector** | Terkoneksi ke node-node penting |

## Pipeline
1. Install & Import
2. Load Data & Validasi
3. Build Network Graph
4. Network Overview Statistics
5. Centrality Metrics Computation
6. Influencer Tier Ranking
7. Community Detection (Louvain)
8. White Space Analysis
9. Export untuk Backend API


---
## 0. Install & Import

In [3]:
# ── Install dependencies (jalankan sekali) ─────────────────────────────────
!pip install networkx python-louvain pyvis pandas numpy matplotlib seaborn scipy -q


[notice] A new release of pip is available: 24.0 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import seaborn as sns
import json
import warnings
from collections import Counter
from pyvis.network import Network
import community as community_louvain

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 120

print('✅ Libraries loaded')

✅ Libraries loaded


In [7]:
# ── Configuration ──────────────────────────────────────────────────────────

# Core Weight Parameters
BASE_COMMENT_WEIGHT         = 1.0    # Base weight untuk setiap comment edge
INTERACTION_WEIGHT_BOOST    = 1.5   # α → Boost jika ada reply
LIKE_WEIGHT_BOOST           = 1.0   # β → Boost dari total like
MENTION_WEIGHT_BOOST        = 1.0    # γ → Boost dari total mention

# Scaling Method
USE_LOG_SCALE               = True   # Gunakan log(1+x) untuk interaction & like

# Edge Filtering
MIN_EDGE_WEIGHT             = 1      # Filter edges di bawah threshold

# Visualization
TOP_N_NODES_VIZ             = 500    # Limit node untuk Pyvis (performa)

# Centrality
PAGERANK_ALPHA              = 0.85

# Community Detection
LOUVAIN_RESOLUTION          = 1.0
LOUVAIN_RANDOM_STATE    = 42

print("✅ Config set (mention + interaction + like weighting)")

✅ Config set (mention + interaction + like weighting)


---
## 1. Load Data & Validasi

In [8]:
# ── Load raw dataset ───────────────────────────────────────────────────────
df = pd.read_csv('../data/processed/dataset_final_clean_revised.csv', low_memory=False)
print(f'Raw dataset        : {df.shape}')
print(f'Columns            : {list(df.columns)}')

Raw dataset        : (2215, 20)
Columns            : ['date', 'keyword', 'url', 'content', 'username', 'total_like', 'total_interaction', 'content_processed', 'sentiment_label', 'sentiment_score', 'post_id', 'comment_id', 'is_comment', 'year_month', 'day_of_week', 'hour', 'mentions', 'hashtags', 'n_mentions', 'n_hashtags']


In [9]:
# ── Load edge table (PRIMARY input untuk SNA) ──────────────────────────────
# Kolom wajib: commenter_list, post_owner_username, comment_count
edge_df = pd.read_csv('../data/processed/post_comment_rank.csv')
print(f'Edge table shape   : {edge_df.shape}')
print(f'Columns            : {list(edge_df.columns)}')
print()
print(edge_df.head(5).to_string(index=False))

Edge table shape   : (190, 10)
Columns            : ['post_id', 'post_owner_username', 'count_post', 'comment_count', 'unique_commenters', 'total_comment_like', 'post_like', 'post_interaction', 'comment_id_list', 'commenter_list']

    post_id post_owner_username  count_post  comment_count  unique_commenters  total_comment_like  post_like  post_interaction                                                                                                                                                                                                    comment_id_list                                                                                                                                                                    commenter_list
DWub4RBAX-U        agam.munawar           1              1                  1                   0         47                 0                                                                                                                              

In [13]:
# ── Validasi kolom wajib ───────────────────────────────────────────────────
REQUIRED = {'commenter_list', 'post_owner_username', 'comment_count'}
missing  = REQUIRED - set(edge_df.columns)
if missing:
    raise ValueError(
        f'❌ Kolom tidak ditemukan: {missing}\n'
        f'   Pastikan output EDA memiliki kolom: {REQUIRED}'
    )
print('✅ Kolom edge valid')
print(f'   Unique commenters  : {edge_df["commenter_list"].nunique():,}')
print(f'   Unique post owners : {edge_df["post_owner_username"].nunique():,}')
print(f'   comment_count range: {edge_df["comment_count"].min()} – {edge_df["comment_count"].max()}')

✅ Kolom edge valid
   Unique commenters  : 189
   Unique post owners : 123
   comment_count range: 1 – 10


In [10]:
# ── Load comments_received (in-degree proxy) ───────────────────────────────
comments_received = pd.read_csv('../data/processed/user_comments_received.csv')
print(f'comments_received shape : {comments_received.shape}')
print(comments_received.head(10).to_string(index=False))

comments_received shape : (123, 2)
          username  total_comments_received
         sindonews                       79
moltensalt.insight                       42
        otoproject                       39
       bitorexpost                       38
           metrotv                       30
       luarbioskop                       29
  koranbaliexpress                       26
   investordailyid                       20
         kompascom                       20
 sukabumiupdatecom                       19


In [11]:
# ── Load mention edges (LAYER 2) ─────────────────────────────────
edges_mention = pd.DataFrame()
try:
    edges_mention = pd.read_csv('../data/processed/edges_mention.csv')
    print(f'✅ Mention edges : {len(edges_mention):,} rows')
except FileNotFoundError:
    print('⚠️  edges_mention.csv tidak ditemukan — mention layer di-skip')

# ── Load keyword data untuk node attribute ──────────────────────────────────
user_keywords = {}
if 'username' in df.columns and 'keyword' in df.columns:
    for user, grp in df.groupby('username')['keyword']:
        user_keywords[user] = list(grp.dropna().unique())
    print(f'✅ Keyword node attributes : {len(user_keywords):,} users')
else:
    print('⚠️  Keyword data tidak tersedia')

✅ Mention edges : 99 rows
✅ Keyword node attributes : 1,642 users


In [14]:
# ── Load comment engagement edges (interaction + like) ─────────────────────

edges_comment = pd.DataFrame()

try:
    edges_comment = pd.read_csv('../data/processed/reply_like_comment.csv')

    print(f'✅ Comment edges : {len(edges_comment):,} rows')

    # Ensure numeric types (penting untuk weighting)

    for col in ["total_interaction", "total_like"]:

        if col in edges_comment.columns:
            edges_comment[col] = (
                edges_comment[col]
                .fillna(0)
                .astype(int)
            )

        else:
            print(f'⚠️  Column {col} tidak ditemukan')

except FileNotFoundError:

    print(
        '⚠️  reply_like_comment.csv tidak ditemukan — comment layer di-skip'
    )

✅ Comment edges : 887 rows


---
## 2. Build Network Graph

- Node = username (post owner maupun commenter)
- Edge = commenter → post_owner, weight = comment_count + mention + total_like + total_interaction


In [ ]:
# ── Build edges using post rows as owner mapping ─────────────────────

import pandas as pd

print("Building edges using post rows as owner mapping...")


# =========================================================
# STEP 1 — Load dataset
# =========================================================

df = pd.read_csv(
    "../data/processed/dataset_final_clean_revised.csv",
    low_memory=False
)

print("Rows:", len(df))


# =========================================================
# STEP 2 — Identify post rows
# =========================================================

df_posts = df[
    df["is_comment"] == False
][
    [
        "post_id",
        "username"
    ]
].drop_duplicates(
    "post_id"
)

df_posts = df_posts.rename(
    columns={
        "username": "post_owner_username"
    }
)

print("Posts:", len(df_posts))


# =========================================================
# STEP 3 — Identify comment rows
# =========================================================

df_comments = df[
    df["is_comment"] == True
].copy()

print("Comments:", len(df_comments))


# =========================================================
# STEP 4 — Merge post owner
# =========================================================

df_comments = df_comments.merge(

    df_posts,

    on="post_id",

    how="left"

)


# =========================================================
# STEP 5 — Clean numeric
# =========================================================

df_comments["total_interaction"] = (

    pd.to_numeric(
        df_comments["total_interaction"],
        errors="coerce"
    )

    .fillna(0)

)


# =========================================================
# STEP 6 — Weight per comment
# =========================================================

df_comments["interaction_weight"] = (

    1

    +

    df_comments["total_interaction"]

)


# =========================================================
# STEP 7 — Aggregate per commenter → post owner
# =========================================================

edges_final = (

    df_comments

    .groupby(
        [
            "username",                 # commenter
            "post_owner_username"
        ],
        as_index=False
    )

    .agg(

        comment_count=(
            "comment_id",
            "count"
        ),

        total_interaction=(
            "total_interaction",
            "sum"
        ),

        weight=(
            "interaction_weight",
            "sum"
        )

    )

)


# =========================================================
# STEP 8 — Rename columns
# =========================================================

edges_final = edges_final.rename(
    columns={
        "username": "source",
        "post_owner_username": "target"
    }
)


edges_final["edge_type"] = "comment"


# =========================================================
# STEP 9 — Remove self-loop
# =========================================================

edges_final = edges_final[
    edges_final["source"]
    !=
    edges_final["target"]
]


print()
print("✅ Edge built")

print("Edges:", len(edges_final))

print()

print(
    edges_final.head()
)

Building edges using post rows as owner mapping...
Rows: 2215
Posts: 1292
Comments: 869

✅ Edge built
Edges: 823

             source          target  comment_count  total_interaction  weight  \
0          0h_gituh       sindonews              1                  0       1   
1          4erielle     trendjktcom              1                  0       1   
2         4l3x42022       sindonews              1                  0       1   
3         5_famiily  theglobeconomy              1                  0       1   
4  76henikusumawati       sindonews              1                  0       1   

  edge_type  
0   comment  
1   comment  
2   comment  
3   comment  
4   comment  


In [41]:
# =========================================================
# STEP 10 — Build Network Graph
# =========================================================

import networkx as nx

print("Building graph...")

G_directed = nx.DiGraph()

# Add nodes
all_nodes = set(
    edges_final["source"]
).union(
    set(edges_final["target"])
)

for node in all_nodes:

    G_directed.add_node(node)

# Add edges

for _, row in edges_final.iterrows():

    G_directed.add_edge(

        row["source"],

        row["target"],

        weight=float(row["weight"])

    )

G_undirected = G_directed.to_undirected()

print()

print("Graph built")

print(
    "Nodes:",
    G_directed.number_of_nodes()
)

print(
    "Edges:",
    G_directed.number_of_edges()
)

Building graph...

Graph built
Nodes: 931
Edges: 823


---
## 3. Network Overview Statistics

In [42]:
# =========================================================
# STEP 11 — Network Overview
# =========================================================

import numpy as np

N = G_directed.number_of_nodes()

E = G_directed.number_of_edges()

density = nx.density(G_directed)

components = list(
    nx.connected_components(
        G_undirected
    )
)

largest_cc = max(
    components,
    key=len
)

lcc_ratio = len(largest_cc) / N

in_degrees = dict(
    G_directed.in_degree()
)

out_degrees = dict(
    G_directed.out_degree()
)

print()

print("=" * 60)

print("NETWORK OVERVIEW")

print("=" * 60)

print(
    "Nodes:",
    N
)

print(
    "Edges:",
    E
)

print(
    "Density:",
    round(density, 6)
)

print(
    "Connected components:",
    len(components)
)

print(
    "Largest CC size:",
    len(largest_cc)
)

print(
    "LCC ratio:",
    round(lcc_ratio, 3)
)


NETWORK OVERVIEW
Nodes: 931
Edges: 823
Density: 0.000951
Connected components: 108
Largest CC size: 90
LCC ratio: 0.097


---
## 4. Centrality Metrics Computation

Semua metric dihitung dari graph `commenter → post_owner` dengan weight = `comment_count`.


In [44]:
# =========================================================
# STEP 12 — Centrality Metrics (Robust Version)
# =========================================================

print()

print("Computing centrality metrics...")

# ---------------------------------------------------------
# Degree
# ---------------------------------------------------------

degree_centrality = nx.degree_centrality(
    G_undirected
)

in_degree_centrality = nx.in_degree_centrality(
    G_directed
)

out_degree_centrality = nx.out_degree_centrality(
    G_directed
)

# ---------------------------------------------------------
# PageRank
# ---------------------------------------------------------

pagerank = nx.pagerank(

    G_directed,

    weight="weight",

    alpha=0.85,

    max_iter=200
)

# ---------------------------------------------------------
# Betweenness
# (sample kalau graph besar)
# ---------------------------------------------------------

N = G_directed.number_of_nodes()

if N > 1000:

    print("Large graph detected — sampling betweenness")

    betweenness = nx.betweenness_centrality(

        G_undirected,

        weight="weight",

        k=500,

        normalized=True

    )

else:

    betweenness = nx.betweenness_centrality(

        G_undirected,

        weight="weight"

    )

# ---------------------------------------------------------
# Closeness
# ---------------------------------------------------------

closeness = nx.closeness_centrality(
    G_undirected
)

# ---------------------------------------------------------
# Eigenvector (robust)
# ---------------------------------------------------------

try:

    eigenvector = nx.eigenvector_centrality(

        G_undirected,

        weight="weight",

        max_iter=1000

    )

    print("Eigenvector converged")

except nx.PowerIterationFailedConvergence:

    print("Eigenvector failed — using numpy solver")

    eigenvector = nx.eigenvector_centrality_numpy(

        G_undirected,

        weight="weight"

    )

print()

print("Centrality done")


Computing centrality metrics...
Eigenvector converged

Centrality done


In [45]:
# =========================================================
# STEP 13 — Centrality Table
# =========================================================

all_nodes = list(
    G_directed.nodes()
)

centrality_df = pd.DataFrame({

    "username":

        all_nodes,

    "in_degree":

        [G_directed.in_degree(n)
         for n in all_nodes],

    "out_degree":

        [G_directed.out_degree(n)
         for n in all_nodes],

    "pagerank":

        [pagerank.get(n, 0)
         for n in all_nodes],

    "betweenness":

        [betweenness.get(n, 0)
         for n in all_nodes],

    "closeness":

        [closeness.get(n, 0)
         for n in all_nodes],

    "eigenvector":

        [eigenvector.get(n, 0)
         for n in all_nodes]

})

print()

print("Centrality table ready")

print(
    centrality_df.head()
)


Centrality table ready
             username  in_degree  out_degree  pagerank  betweenness  \
0             abchopq          0           1  0.000618     0.000000   
1  moltensalt.insight         18           0  0.010080     0.000354   
2       geodipaenergi          0           1  0.000618     0.000000   
3         rbmohshaleh          0           1  0.000618     0.000000   
4         nation_027_          0           1  0.000618     0.000000   

   closeness  eigenvector  
0   0.015865          0.0  
1   0.019355          0.0  
2   0.004411          0.0  
3   0.005659          0.0  
4   0.043455          0.0  


In [ ]:
# =========================================================
# STEP 14 — Community Detection
# =========================================================

import community as community_louvain

print()

print("Running Louvain...")

partition = community_louvain.best_partition(

    G_undirected,

    weight="weight",

    random_state=42

)

n_communities = len(

    set(
        partition.values()
    )
)

centrality_df["community_id"] = (

    centrality_df["username"]

    .map(partition)

)

print()

print(
    "Communities:",
    n_communities
)

✅ Centrality DataFrame: (435, 20)
                                                   username  in_degree  out_degree  total_degree  in_degree_centrality  out_degree_centrality  degree_centrality  betweenness_centrality  closeness_centrality  pagerank  eigenvector_centrality     hub_score  authority_score                 active_keywords  post_count  total_comments_recv  total_post_like  total_interaction  avg_interaction  comments_received_raw
0                                        moltensalt.insight          5           0             5              0.011521               0.000000           0.011521                0.000106                   0.0  0.008127            3.835516e-18  0.000000e+00     0.000000e+00              ["TransisiEnergi"]         5.0                 42.0            127.0                0.0              0.0                   42.0
1  ['febriyvn.me', 'gsitest', 'pln.uipklb', 'purwanindita']          0           1             1              0.000000               0.00230

---
## 5. Influencer Tier Ranking

**Composite Influence Score:**

| Metric | Weight | Alasan |
|---|---|---|
| PageRank | 30% | Komentar dari akun besar = lebih bernilai |
| Betweenness | 25% | Broker antar komunitas → high campaign value |
| In-degree centrality | 20% | Raw popularity — berapa akun yang datang |
| Authority Score | 15% | Dipilih oleh hub-hub besar |
| Eigenvector | 10% | Terkoneksi ke node penting |


In [47]:
# =========================================================
# STEP 15 — Influence Score
# =========================================================

def normalize(series):

    return (

        series - series.min()

    ) / (

        series.max() - series.min()

    )


centrality_df["pagerank_norm"] = normalize(
    centrality_df["pagerank"]
)

centrality_df["betweenness_norm"] = normalize(
    centrality_df["betweenness"]
)

centrality_df["in_degree_norm"] = normalize(
    centrality_df["in_degree"]
)

centrality_df["eigenvector_norm"] = normalize(
    centrality_df["eigenvector"]
)

centrality_df["influence_score"] = (

    0.3 * centrality_df["pagerank_norm"]

    +

    0.25 * centrality_df["betweenness_norm"]

    +

    0.2 * centrality_df["in_degree_norm"]

    +

    0.25 * centrality_df["eigenvector_norm"]

)

centrality_df = centrality_df.sort_values(

    "influence_score",

    ascending=False

)

print()

print("Top Influencers")

print(

    centrality_df.head(20)
)


Top Influencers
               username  in_degree  out_degree  pagerank  betweenness  \
138           sindonews         78           0  0.040831     0.008986   
715         bitorexpost         38           0  0.020593     0.001627   
226          otoproject         38           0  0.020436     0.002160   
216             metrotv         30           0  0.016388     0.001007   
742         luarbioskop         29           0  0.015862     0.000940   
581     investordailyid         19           0  0.010152     0.002951   
559          ronnypunya          0           1  0.000618     0.000000   
6      koranbaliexpress         24           0  0.013234     0.000639   
286           kompascom         20           0  0.011131     0.000440   
35    sukabumiupdatecom         19           0  0.010606     0.000396   
627            bi_lang_          0           1  0.000618     0.000000   
1    moltensalt.insight         18           0  0.010080     0.000354   
576        katadatacoid         16

In [48]:
# =========================================================
# STEP 16 — Tier Assignment
# =========================================================

def assign_tier(score):

    if score >= 0.8:

        return "Mega"

    elif score >= 0.6:

        return "Macro"

    elif score >= 0.4:

        return "Mid"

    elif score >= 0.2:

        return "Micro"

    else:

        return "Regular"


centrality_df["tier"] = (

    centrality_df["influence_score"]

    .apply(assign_tier)

)

print()

print(

    centrality_df["tier"]

    .value_counts()
)


tier
Regular    925
Micro        4
Mid          1
Macro        1
Name: count, dtype: int64


In [49]:
# =========================================================
# Helper Functions
# =========================================================

import json
from collections import Counter

def min_max_normalize(series):

    if series.max() == series.min():

        return series * 0

    return (

        series - series.min()

    ) / (

        series.max() - series.min()

    )

In [50]:
# ── Business Question Matrix ─────────────────────────────

print('=' * 65)
print('  Q: Siapa influencer utama? (PageRank + In-degree)')
print('=' * 65)

print(

    centrality_df

    .sort_values(
        'influence_score',
        ascending=False
    )

    [[
        'username',
        'tier',
        'in_degree',
        'pagerank',
        'influence_score'
    ]]

    .head(10)

    .to_string(index=False)

)


print()
print('=' * 65)
print('  Q: Siapa broker antar komunitas? (Betweenness)')
print('=' * 65)

print(

    centrality_df

    .sort_values(
        'betweenness',
        ascending=False
    )

    [[
        'username',
        'tier',
        'betweenness',
        'in_degree',
        'influence_score'
    ]]

    .head(10)

    .to_string(index=False)

)


print()
print('=' * 65)
print('  Q: Siapa yang paling aktif komentar? (Out-degree)')
print('=' * 65)

print(

    centrality_df

    .sort_values(
        'out_degree',
        ascending=False
    )

    [[
        'username',
        'tier',
        'out_degree',
        'in_degree',
        'influence_score'
    ]]

    .head(10)

    .to_string(index=False)

)

  Q: Siapa influencer utama? (PageRank + In-degree)
         username    tier  in_degree  pagerank  influence_score
        sindonews   Macro         78  0.040831         0.750000
      bitorexpost     Mid         38  0.020593         0.541729
       otoproject   Micro         38  0.020436         0.305364
          metrotv   Micro         30  0.016388         0.222584
      luarbioskop   Micro         29  0.015862         0.214231
  investordailyid   Micro         19  0.010152         0.201951
       ronnypunya Regular          0  0.000618         0.179624
 koranbaliexpress Regular         24  0.013234         0.173430
        kompascom Regular         20  0.011131         0.141949
sukabumiupdatecom Regular         19  0.010606         0.134240

  Q: Siapa broker antar komunitas? (Betweenness)
              username    tier  betweenness  in_degree  influence_score
             sindonews   Macro     0.008986         78         0.750000
       investordailyid   Micro     0.002951       

---
## 6. Community Detection (Louvain)

In [51]:
# ── Louvain ─────────────────────────────────────────────

import community as community_louvain

print('Running Louvain...')

LOUVAIN_RESOLUTION = 1.0
LOUVAIN_RANDOM_STATE = 42

partition = community_louvain.best_partition(

    G_undirected,

    weight='weight',

    resolution=LOUVAIN_RESOLUTION,

    random_state=LOUVAIN_RANDOM_STATE,

)

n_communities = len(

    set(
        partition.values()
    )
)

modularity = community_louvain.modularity(

    partition,

    G_undirected,

    weight='weight'

)

print(f'✅ Communities  : {n_communities}')

print(
    f'   Modularity   : {modularity:.4f}'
)

centrality_df['community_id'] = (

    centrality_df['username']

    .map(partition)

)

community_sizes = (

    pd.Series(partition)

    .value_counts()

    .sort_values(
        ascending=False
    )

)

print()
print('Top 15 communities by size:')

print(
    community_sizes
    .head(15)
    .to_string()
)

Running Louvain...
✅ Communities  : 109
   Modularity   : 0.9261

Top 15 communities by size:
25    90
8     45
5     39
46    34
11    31
0     30
54    29
6     25
32    21
81    20
10    20
1     19
17    17
21    16
77    16


In [52]:
# ── Community Profiling ─────────────────────────────────

community_profiles = []

for comm_id, size in community_sizes.items():

    comm_users = centrality_df[

        centrality_df['community_id']
        ==
        comm_id

    ]

    if comm_users.empty:

        continue


    top_influencer = (

        comm_users

        .sort_values(
            'influence_score',
            ascending=False
        )

        .iloc[0]

    )


    top_broker = (

        comm_users

        .sort_values(
            'betweenness',
            ascending=False
        )

        .iloc[0]

    )


    top_commenter = (

        comm_users

        .sort_values(
            'out_degree',
            ascending=False
        )

        .iloc[0]

    )


    comm_posts = df[

        df['username']

        .isin(
            comm_users['username']
        )

    ]


    dominant_sentiment = 'unknown'
    dominant_keyword   = 'unknown'


    if len(comm_posts) > 0:

        if 'sentiment_label' in comm_posts.columns:

            dominant_sentiment = (

                comm_posts[
                    'sentiment_label'
                ]

                .mode()

                .iloc[0]

            )


        if 'keyword' in comm_posts.columns:

            dominant_keyword = (

                comm_posts[
                    'keyword'
                ]

                .mode()

                .iloc[0]

            )


    community_profiles.append({

        'community_id':

            int(comm_id),

        'size':

            int(size),

        'top_influencer':

            top_influencer[
                'username'
            ],

        'top_broker':

            top_broker[
                'username'
            ],

        'top_commenter':

            top_commenter[
                'username'
            ],

        'avg_influence_score':

            round(

                float(

                    comm_users[
                        'influence_score'
                    ].mean()

                ),

                6

            ),

        'dominant_sentiment':

            dominant_sentiment,

        'dominant_keyword':

            dominant_keyword,

    })


community_profiles_df = (

    pd.DataFrame(
        community_profiles
    )

    .sort_values(
        'size',
        ascending=False
    )

)


print()
print('Community Profiles (Top 10):')

print(

    community_profiles_df

    [[
        'community_id',
        'size',
        'top_influencer',
        'top_broker',
        'dominant_keyword'
    ]]

    .head(10)

    .to_string(index=False)

)


Community Profiles (Top 10):
 community_id  size   top_influencer       top_broker dominant_keyword
           25    90        sindonews        sindonews   ProduksiMinyak
            8    45       otoproject       otoproject   ProduksiMinyak
            5    39      bitorexpost      bitorexpost   ProduksiMinyak
           46    34    indovibing.id      tututata151   ProduksiMinyak
           11    31          metrotv          metrotv   ProduksiMinyak
            0    30      luarbioskop      luarbioskop   ProduksiMinyak
           54    29  investordailyid  investordailyid   ProduksiMinyak
            6    25 koranbaliexpress koranbaliexpress   ProduksiMinyak
           32    21        kompascom        kompascom   ProduksiMinyak
           81    20          zonaebt          cfdhack   ProduksiMinyak


In [54]:
# =========================================================
# FIX — Ensure tier numeric
# =========================================================

if centrality_df["tier"].dtype == object:

    tier_map = {
        "Mega": 1,
        "Macro": 2,
        "Mid": 3,
        "Micro": 4,
        "Regular": 5
    }

    centrality_df["tier_numeric"] = (
        centrality_df["tier"]
        .map(tier_map)
        .fillna(5)
        .astype(int)
    )

else:

    centrality_df["tier_numeric"] = (
        centrality_df["tier"]
        .astype(int)
    )

---
## 7. White Space Analysis

White space = area **under-served** tapi punya potensi tinggi:
1. Keyword dengan engagement tinggi tapi sedikit influencer aktif → *opportunity gap*
2. Keyword dengan sentimen negatif dominan → *unmet needs*
3. Komunitas terisolasi tapi aktif → *untapped audience*


In [55]:
# ── White Space: Keyword Opportunity ─────────────────────

whitespace_results = {}

if 'keyword' in df.columns:

    kw_analysis = (

        df

        .groupby('keyword')

        .agg(

            total_posts = (

                'username',

                'count'

            ),

            unique_users = (

                'username',

                'nunique'

            ),

            avg_interaction = (

                'total_interaction',

                'mean'

            )

        )

        .reset_index()

    )


    kw_analysis['interaction_per_user'] = (

        kw_analysis[
            'avg_interaction'
        ]

        /

        kw_analysis[
            'unique_users'
        ].clip(lower=1)

    )


    tier12 = set(

        centrality_df[

            centrality_df[
                'tier_numeric'
            ]

            <= 2

        ]['username']

    )


    kw_analysis['influencer_count'] = (

        kw_analysis['keyword']

        .map(

            df[

                df['username']

                .isin(
                    tier12
                )

            ]

            .groupby('keyword')

            ['username']

            .nunique()

        )

        .fillna(0)

        .astype(int)

    )


    kw_analysis['whitespace_score'] = (

        min_max_normalize(

            kw_analysis[
                'interaction_per_user'
            ]

        )

        -

        min_max_normalize(

            kw_analysis[
                'influencer_count'
            ]

        )

    ).clip(lower=0)


    print()
    print('Keyword White Space:')

    print(

        kw_analysis

        .sort_values(
            'whitespace_score',
            ascending=False
        )

        .head(20)

        .to_string(index=False)

    )


    whitespace_results['keyword_analysis'] = (

        kw_analysis

        .to_dict('records')

    )


Keyword White Space:
         keyword  total_posts  unique_users  avg_interaction  interaction_per_user  influencer_count  whitespace_score
 HargaMinyakBumi            1             1         0.000000              0.000000                 0               0.0
KenaikanhargaBBM           36            26         0.000000              0.000000                 0               0.0
     KonsumsiBBM            6             6         0.000000              0.000000                 1               0.0
    KrisisEnergi          500           399         0.000000              0.000000                 1               0.0
 MinyakIndonesia           15            14         0.000000              0.000000                 0               0.0
  ProduksiMinyak          879           820         1.878271              0.002291                 1               0.0
      SubsidiBBM          278           234         0.000000              0.000000                 0               0.0
  TransisiEnergi          

In [56]:
# ── White Space: Community Isolation ─────────────────────

isolated_communities = []

for comm_id in community_sizes.index:

    comm_nodes = [

        n

        for n, c in partition.items()

        if c == comm_id

    ]


    if len(comm_nodes) < 2:

        continue


    subgraph = G_undirected.subgraph(

        comm_nodes

    )


    internal_edges = (

        subgraph

        .number_of_edges()

    )


    half_degree = (

        sum(

            G_undirected.degree(n)

            for n in comm_nodes

        )

        /

        2

    )


    isolation_ratio = (

        internal_edges

        /

        max(
            half_degree,
            1
        )

    )


    isolated_communities.append({

        'community_id':

            int(comm_id),

        'size':

            int(

                community_sizes[
                    comm_id
                ]

            ),

        'isolation_ratio':

            round(
                isolation_ratio,
                4
            ),

        'is_isolated':

            isolation_ratio > 0.85,

    })


isolated_df = pd.DataFrame(

    isolated_communities

)


print()
print(
    'Highly isolated communities:',
    isolated_df[
        'is_isolated'
    ].sum()
)


whitespace_results[
    'isolated_communities'
] = isolated_df.to_dict(
    'records'
)


Highly isolated communities: 109


---
## 8. Export untuk Backend API

Format: **node-link JSON** compatible dengan D3.js / react-force-graph / sigma.js

Setiap node include `active_keywords` → dipakai untuk filter & warna di website UI.


In [ ]:
# =========================================================
# STEP 17 — Export
# =========================================================

centrality_df.to_csv(

    "../data/processed/influencer_ranking.csv",

    index=False

)

edges_final.to_csv(

    "../data/processed/network_edges.csv",

    index=False

)

print()

print("Export complete")

In [29]:
# ── Pyvis Interactive Preview (Robust Version) ──────────────────────────────

import json
import pandas as pd
from pyvis.network import Network

# =========================================================
# Helper functions (SAFE CONVERSION)
# =========================================================

def safe_int(x):
    try:
        if pd.isna(x):
            return 0
        return int(float(x))
    except:
        return 0

def safe_float(x):
    try:
        if pd.isna(x):
            return 0.0
        return float(x)
    except:
        return 0.0


# =========================================================
# Select top nodes
# =========================================================

top_nodes = (
    centrality_df
    .sort_values('influence_score', ascending=False)
    .head(TOP_N_NODES_VIZ)['username']
    .tolist()
)

G_viz = G_undirected.subgraph(top_nodes).copy()


# =========================================================
# Color palette
# =========================================================

COMMUNITY_COLORS = [
    '#E74C3C','#3498DB','#2ECC71','#F39C12','#9B59B6',
    '#1ABC9C','#E67E22','#34495E','#F1C40F','#16A085',
    '#8E44AD','#27AE60','#D35400','#2980B9','#C0392B',
    '#A93226','#148F77','#B7950B','#6C3483','#117A65',
]


# =========================================================
# Initialize Pyvis
# =========================================================

net = Network(
    height="750px",
    width="100%",
    bgcolor="#1a1a2e",
    font_color="white",
    directed=False,
)

net.barnes_hut(
    gravity=-8000,
    central_gravity=0.3,
    spring_length=100,
    spring_strength=0.05,
)


# =========================================================
# PERFORMANCE: convert dataframe to dict lookup
# =========================================================

row_lookup = (
    centrality_df
    .set_index("username")
    .to_dict("index")
)


# =========================================================
# Add nodes
# =========================================================

for node in G_viz.nodes():

    row = row_lookup.get(node)

    if not row:
        continue

    comm_id = safe_int(row.get("community_id", 0))
    tier    = safe_int(row.get("tier", 5))

    influence_score = safe_float(
        row.get("influence_score", 0)
    )

    size = 8 + influence_score * 60

    color = COMMUNITY_COLORS[
        comm_id % len(COMMUNITY_COLORS)
    ]

    # keywords
    try:
        kw_display = ", ".join(
            json.loads(
                str(row.get("active_keywords", "[]"))
            )[:5]
        )
    except:
        kw_display = ""

    # =====================================================
    # Tooltip
    # =====================================================

    title = (
        f"<b>@{node}</b><br>"
        f"Tier            : {row.get('tier_label', '?')}<br>"
        f"Influence Score : {influence_score:.4f}<br>"
        f"PageRank        : {safe_float(row.get('pagerank')):.6f}<br>"
        f"Betweenness     : {safe_float(row.get('betweenness_centrality')):.6f}<br>"
        f"In-degree       : {safe_int(row.get('in_degree'))}<br>"
        f"Out-degree      : {safe_int(row.get('out_degree'))}<br>"
        f"Comments recv   : {safe_int(row.get('total_comments_recv')):,}<br>"
        f"Posts           : {safe_int(row.get('post_count'))}<br>"
        f"Community       : #{comm_id}<br>"
        f"Keywords        : {kw_display}"
    )

    net.add_node(
        node,
        label=node if tier <= 3 else "",
        title=title,
        size=max(size, 5),
        color=color,
        borderWidth=3 if tier == 1 else (2 if tier == 2 else 1),
        borderWidthSelected=5,
    )


# =========================================================
# Add edges
# =========================================================

for u, v, data in G_viz.edges(data=True):

    weight = safe_float(
        data.get("weight", 1)
    )

    net.add_edge(
        u,
        v,
        value=min(weight, 10),
        color="rgba(255,255,255,0.08)",
    )


# =========================================================
# Save visualization
# =========================================================

viz_path = "../data/processed/network_visualization.html"

net.save_graph(viz_path)

print(f"✅ Pyvis saved: {viz_path}")
print(
    f"   Nodes: {G_viz.number_of_nodes():,}  "
    f"Edges: {G_viz.number_of_edges():,}"
)

✅ Pyvis saved: ../data/processed/network_visualization.html
   Nodes: 435  Edges: 288


In [31]:
# ── Export JSON untuk Backend (Robust Version) ─────────────────────────────

import json
import pandas as pd
from collections import Counter
import networkx as nx

# =========================================================
# Helper functions
# =========================================================

def safe_int(x):
    try:
        if pd.isna(x):
            return 0
        return int(float(x))
    except:
        return 0

def safe_float(x):
    try:
        if pd.isna(x):
            return 0.0
        return float(x)
    except:
        return 0.0

def safe_keywords(x):
    try:
        if pd.isna(x):
            return []
        return json.loads(str(x))
    except:
        return []


# =========================================================
# PERFORMANCE: build lookup dictionary
# =========================================================

row_lookup = (
    centrality_df
    .set_index("username")
    .to_dict("index")
)

nodes_export = []

for node in G_directed.nodes():

    row = row_lookup.get(node)

    if not row:
        continue

    kw_list = safe_keywords(
        row.get("active_keywords")
    )

    nodes_export.append({

        'id': node,
        'username': node,

        # Influence
        'influence_score':
            round(safe_float(row.get('influence_score')), 6),

        'tier':
            safe_int(row.get('tier', 5)),

        'tier_label':
            str(row.get('tier_label', 'Regular User')),

        # Centrality
        'pagerank':
            round(safe_float(row.get('pagerank')), 8),

        'betweenness':
            round(safe_float(row.get('betweenness_centrality')), 8),

        'closeness':
            round(safe_float(row.get('closeness_centrality')), 8),

        'eigenvector':
            round(safe_float(row.get('eigenvector_centrality')), 8),

        'authority_score':
            round(safe_float(row.get('authority_score')), 8),

        'hub_score':
            round(safe_float(row.get('hub_score')), 8),

        # Degree
        'in_degree':
            safe_int(row.get('in_degree')),

        'out_degree':
            safe_int(row.get('out_degree')),

        'total_degree':
            safe_int(row.get('total_degree')),

        # Content metrics
        'post_count':
            safe_int(row.get('post_count')),

        'total_comments_recv':
            safe_int(row.get('total_comments_recv')),

        'total_interaction':
            safe_float(row.get('total_interaction')),

        'avg_interaction':
            round(safe_float(row.get('avg_interaction')), 2),

        'total_post_like':
            safe_float(row.get('total_post_like')),

        # Community
        'community_id':
            safe_int(row.get('community_id')),

        # Keywords
        'active_keywords':
            kw_list,
    })


# =========================================================
# Export edges
# =========================================================

edges_export = []

for u, v, data in G_directed.edges(data=True):

    edges_export.append({

        'source': u,

        'target': v,

        'weight':
            round(
                safe_float(
                    data.get("weight", 1)
                ),
                4
            ),

        'edge_type':
            str(data.get("edge_type", "comment")),
    })


# =========================================================
# Metadata
# =========================================================

edge_type_counts = Counter(
    e["edge_type"]
    for e in edges_export
)

graph_data = {

    'metadata': {

        'total_nodes':
            G_directed.number_of_nodes(),

        'total_edges':
            G_directed.number_of_edges(),

        'density':
            round(
                nx.density(G_directed),
                8
            ),

        'modularity':
            round(modularity, 6),

        'n_communities':
            n_communities,

        'lcc_ratio':
            round(lcc_ratio, 4),

        'edge_architecture':
            'comment-on-post (PRIMARY) + mention (LAYER 2)',

        'edge_type_counts':
            dict(edge_type_counts),
    },

    'nodes': nodes_export,

    'edges': edges_export,

    'communities':
        community_profiles_df.to_dict('records'),

    'whitespace':
        whitespace_results,
}


# =========================================================
# Save JSON
# =========================================================

output_path = "../data/processed/graph_data.json"

with open(output_path, "w") as f:

    json.dump(
        graph_data,
        f,
        default=str,
        ensure_ascii=False
    )

print("✅ graph_data.json exported")

print(
    f"Nodes : {len(nodes_export):,}"
)

print(
    f"Edges : {len(edges_export):,}"
)

✅ graph_data.json exported
Nodes : 435
Edges : 288


In [32]:
# ── Export CSV ───────────────────────────────────────────────────────────────
export_cols = [
    'username', 'influence_score', 'influence_percentile', 'tier', 'tier_label',
    'community_id',
    'in_degree', 'out_degree', 'total_degree',
    'in_degree_centrality', 'out_degree_centrality',
    'pagerank', 'betweenness_centrality', 'closeness_centrality',
    'eigenvector_centrality', 'authority_score', 'hub_score',
    'post_count', 'total_comments_recv', 'total_interaction',
    'avg_interaction', 'total_post_like', 'active_keywords',
]
export_cols = [c for c in export_cols if c in centrality_df.columns]

(
    centrality_df
    .sort_values('influence_score', ascending=False)
    [export_cols]
    .to_csv('../data/processed/influencer_ranking.csv', index=False)
)
community_profiles_df.to_csv('../data/processed/community_profiles.csv', index=False)

print('✅ Exports:')
print(f'   influencer_ranking.csv  ({len(centrality_df):,} users)')
print(f'   community_profiles.csv  ({len(community_profiles_df):,} communities)')
print(f'   graph_data.json')
print(f'   network_visualization.html')

✅ Exports:
   influencer_ranking.csv  (435 users)
   community_profiles.csv  (149 communities)
   graph_data.json
   network_visualization.html


In [33]:
# ── Final Summary ────────────────────────────────────────────────────────────
print('''
╔══════════════════════════════════════════════════════════════════╗
║                  SNA ANALYSIS COMPLETE ✅                        ║
╚══════════════════════════════════════════════════════════════════╝
''')
print(f'  Nodes              : {G_directed.number_of_nodes():,}')
print(f'  Edges              : {G_directed.number_of_edges():,}')
print(f'  Edge architecture  : comment-on-post (weight = comment_count)')
print(f'  Communities        : {n_communities}')
print(f'  Modularity         : {modularity:.4f}')
print()
for tier_id, label in TIER_LABELS.items():
    cnt = (centrality_df['tier'] == tier_id).sum()
    print(f'  Tier {tier_id} ({label:22s}): {cnt:,}')
print()
print('  OUTPUTS:')
print('  • graph_data.json             → D3 / react-force-graph ready')
print('  • influencer_ranking.csv      → per-user centrality + tier + keywords')
print('  • community_profiles.csv      → community summary')
print('  • network_visualization.html  → Pyvis preview')
print()
print('  node.active_keywords → pakai untuk filter & warna di website UI')
print('  node.in_degree       → proxy total komentar diterima')
print('  node.betweenness     → broker score untuk brand outreach')


╔══════════════════════════════════════════════════════════════════╗
║                  SNA ANALYSIS COMPLETE ✅                        ║
╚══════════════════════════════════════════════════════════════════╝

  Nodes              : 435
  Edges              : 288
  Edge architecture  : comment-on-post (weight = comment_count)
  Communities        : 149
  Modularity         : 0.9733

  Tier 1 (Mega Influencer       ): 5
  Tier 2 (Macro Influencer      ): 17
  Tier 3 (Mid Influencer        ): 35
  Tier 4 (Micro Influencer      ): 161
  Tier 5 (Regular User          ): 217

  OUTPUTS:
  • graph_data.json             → D3 / react-force-graph ready
  • influencer_ranking.csv      → per-user centrality + tier + keywords
  • community_profiles.csv      → community summary
  • network_visualization.html  → Pyvis preview

  node.active_keywords → pakai untuk filter & warna di website UI
  node.in_degree       → proxy total komentar diterima
  node.betweenness     → broker score untuk brand outrea